# HEST Dataset Exploration and Pipeline

## Breast Cancer Grading Markers:
- **ESR1** (ER), **PGR** (PR), **ERBB2** (HER2), **MKI67** (Ki67), **TP53**, **EGFR**, Cytokeratins

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import h5py
import scanpy as sc
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import plotly.express as px
import networkx as nx
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import umap
import openslide

sys.path.append('/app/Clustering_Pipeline/Clustering-Pipeline/project')
try:
    from feature_extractor import setup_model_and_transform
    print("Successfully imported feature_extractor")
except ImportError as e:
    print(f"Error importing feature_extractor: {e}")

BREAST_CANCER_MARKERS = {
    'ESR1': 'Estrogen Receptor (ER)',
    'PGR': 'Progesterone Receptor (PR)',
    'ERBB2': 'HER2',
    'MKI67': 'Ki67 Proliferation Marker',
    'TP53': 'Tumor Suppressor P53',
    'EGFR': 'Epidermal Growth Factor Receptor',
    'KRT5': 'Cytokeratin 5',
    'KRT14': 'Cytokeratin 14',
    'KRT19': 'Cytokeratin 19',
    'BRCA1': 'BRCA1',
    'BRCA2': 'BRCA2'
}

PATCH_SIZE = 224
PATCH_SPACING = 470
ROI_DISTANCE_THRESHOLD = PATCH_SPACING * 2 + 10
N_CLUSTERS = 8
MIN_PATCHES_PER_ROI = 20
ROI_TARGET_SIZE = 512  # Target size for ROI images

CONFIG = {
    'DATA_DIR': '../hest_data',
    'OUTPUT_DIR': '../output',
    'SLIDES': ['TENX96', 'TENX99'],
    'DEVICE': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'ROI_DISTANCE_THRESHOLD': ROI_DISTANCE_THRESHOLD,
    'N_CLUSTERS': N_CLUSTERS,
    'BREAST_CANCER_MARKERS': BREAST_CANCER_MARKERS,
    'PATCH_SIZE': PATCH_SIZE,
    'MIN_PATCHES_PER_ROI': MIN_PATCHES_PER_ROI,
    'ROI_TARGET_SIZE': ROI_TARGET_SIZE
}

for slide in CONFIG['SLIDES']:
    for subfolder in ['transcriptomic', 'features', 'graphs', 'roi_images']:
        os.makedirs(os.path.join(CONFIG['OUTPUT_DIR'], slide, subfolder), exist_ok=True)

print(f"Using device: {CONFIG['DEVICE']}")
print(f"ROI target size: {CONFIG['ROI_TARGET_SIZE']}x{CONFIG['ROI_TARGET_SIZE']}")

## 1. Data Loader

In [ ]:
class HESTDataLoader:
    def __init__(self, data_dir):
        self.data_dir = data_dir

    def load_patches_metadata(self, slide_name):
        h5_path = os.path.join(self.data_dir, 'patches', f'{slide_name}.h5')
        if not os.path.exists(h5_path):
            return None, None
        with h5py.File(h5_path, 'r') as f:
            coords = f['coords'][:] if 'coords' in f else None
            barcodes = f['barcode'][:] if 'barcode' in f else None
            if barcodes is not None:
                barcodes = [b[0].decode() if isinstance(b[0], bytes) else str(b[0]) for b in barcodes]
            print(f"[{slide_name}] Patches: {len(coords) if coords is not None else 0}")
            return coords, barcodes

    def load_transcriptomics(self, slide_name):
        h5ad_path = os.path.join(self.data_dir, 'st', f'{slide_name}.h5ad')
        if not os.path.exists(h5ad_path):
            return None
        adata = sc.read_h5ad(h5ad_path)
        print(f"[{slide_name}] AnnData: {adata.shape}")
        return adata
    
    def get_patches_path(self, slide_name):
        return os.path.join(self.data_dir, 'patches', f'{slide_name}.h5')
    
    def get_wsi_path(self, slide_name):
        return os.path.join(self.data_dir, 'wsis', f'{slide_name}.tif')

## 2. Visualizer

In [ ]:
class HESTVisualizer:
    @staticmethod
    def plot_spatial_spots(adata, slide_name):
        if adata is None or 'spatial' not in adata.obsm: return
        coords = adata.obsm['spatial']
        fig = px.scatter(x=coords[:, 0], y=coords[:, 1], title=f'Spots - {slide_name}', width=800, height=800)
        fig.update_traces(marker=dict(size=3))
        fig.update_yaxes(autorange="reversed")
        fig.show()

    @staticmethod
    def plot_umap(embedding, labels, slide_name):
        fig = px.scatter(x=embedding[:, 0], y=embedding[:, 1], color=labels.astype(str),
                         title=f'UMAP - {slide_name}', labels={'color': 'Cluster'})
        fig.show()

    @staticmethod
    def plot_clusters_spatial(coords, labels, slide_name):
        fig = px.scatter(x=coords[:, 0], y=coords[:, 1], color=labels.astype(str),
                         title=f'Clusters - {slide_name}', width=800, height=800)
        fig.update_yaxes(autorange="reversed")
        fig.show()
    
    @staticmethod
    def plot_rois_spatial(coords, rois, slide_name):
        cluster_assignment = np.full(len(coords), -1)
        roi_assignment = np.full(len(coords), -1)
        for roi in rois:
            for idx in roi['patch_indices']:
                cluster_assignment[idx] = roi['cluster_id']
                roi_assignment[idx] = roi['roi_id']
        df = pd.DataFrame({'x': coords[:, 0], 'y': coords[:, 1], 'cluster': cluster_assignment, 'roi_id': roi_assignment})
        df_plot = df[df['roi_id'] != -1].copy()
        fig = px.scatter(df_plot, x='x', y='y', color=df_plot['cluster'].astype(str),
                         hover_data=['roi_id', 'cluster'], title=f'ROIs - {slide_name} ({len(rois)} ROIs)',
                         width=800, height=800, labels={'color': 'Cluster'})
        fig.update_yaxes(autorange="reversed")
        fig.show()

## 3. Feature Extractor

In [ ]:
class HESTFeatureExtractor:
    def __init__(self, device):
        self.device = device
        self.model = None
        self.transform = None
        self._initialized = False

    def _setup_model(self):
        if self._initialized: return
        if self.device.type == 'cuda':
            try:
                self.model, self.transform = setup_model_and_transform(device="cuda")
                print("Using H-optimus model")
            except:
                self._setup_resnet()
        else:
            self._setup_resnet()
        self._initialized = True

    def _setup_resnet(self):
        print("Using ResNet50 model")
        self.device = torch.device("cpu")
        model = models.resnet50(pretrained=True)
        self.model = torch.nn.Sequential(*list(model.children())[:-1])
        self.model.to(self.device).eval()
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def extract(self, h5_path, output_dir, slide_name):
        features_path = os.path.join(output_dir, slide_name, 'features', 'features.npy')
        coords_path = os.path.join(output_dir, slide_name, 'features', 'coords.npy')
        if os.path.exists(features_path) and os.path.exists(coords_path):
            print(f"[{slide_name}] Loading cached features")
            return np.load(features_path), np.load(coords_path)
        self._setup_model()
        print(f"Extracting features for {slide_name}...")
        features_list = []
        with h5py.File(h5_path, 'r') as f:
            imgs, coords = f['img'], f['coords'][:]
            for i in tqdm(range(0, len(imgs), 32)):
                batch = [self.transform(Image.fromarray(img.astype('uint8'))) for img in imgs[i:i+32]]
                if batch:
                    with torch.no_grad():
                        feats = self.model(torch.stack(batch).to(self.device))
                        features_list.append(feats.view(feats.size(0), -1).cpu().numpy())
        if features_list:
            all_features = np.concatenate(features_list, axis=0)
            np.save(features_path, all_features)
            np.save(coords_path, coords)
            return all_features, coords
        return None, None

## 4. Clusterer

In [ ]:
class HESTClusterer:
    def __init__(self, n_clusters=8):
        self.n_clusters = n_clusters

    def run(self, features):
        print(f"Running KMeans with k={self.n_clusters}...")
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init='auto')
        return kmeans.fit_predict(features_scaled)

    def compute_umap(self, features, labels):
        print("Computing UMAP...")
        reducer = umap.UMAP(n_components=2, random_state=42)
        idx = np.random.choice(len(features), min(len(features), 5000), replace=False)
        return reducer.fit_transform(features[idx]), labels[idx]

## 5. Transcriptomics Analyzer

In [ ]:
class HESTTranscriptomicsAnalyzer:
    def __init__(self, markers):
        self.markers = markers
    
    def find_available_markers(self, adata):
        if adata is None: return {}
        available = {g: d for g, d in self.markers.items() if g in adata.var_names}
        print(f"Found {len(available)}/{len(self.markers)} markers")
        return available
    
    def get_marker_expression(self, adata, spot_idx, markers):
        if adata is None or not markers: return {}
        expr = {}
        for gene in markers:
            try:
                val = adata[spot_idx, gene].X
                expr[gene] = float(val.toarray()[0,0] if hasattr(val, 'toarray') else val[0,0])
            except: expr[gene] = 0.0
        return expr
    
    def map_patches_to_spots(self, barcodes, adata):
        if adata is None or barcodes is None: return {}
        spots = adata.obs_names.tolist()
        mapping = {i: spots.index(b) for i, b in enumerate(barcodes) if b in spots}
        print(f"Mapped {len(mapping)}/{len(barcodes)} patches to spots")
        return mapping

## 6. ROI Extractor (Proportional Scale + Padding)

In [ ]:
class HESTROIExtractor:
    """
    Extracts ROI from WSI, scales proportionally to fit target size,
    then adds padding to reach exactly target_size x target_size.
    
    This preserves cell shapes (no distortion) while standardizing size.
    """
    def __init__(self, patch_size=224, target_size=512):
        self.patch_size = patch_size
        self.target_size = target_size
        self.slide = None
        self.current_wsi_path = None
    
    def load_wsi(self, wsi_path):
        if self.current_wsi_path != wsi_path:
            print(f"Loading WSI with OpenSlide: {wsi_path}")
            self.slide = openslide.OpenSlide(wsi_path)
            self.current_wsi_path = wsi_path
            print(f"WSI dimensions: {self.slide.dimensions}")
    
    def compute_bounding_box(self, coords, patch_indices):
        roi_coords = coords[patch_indices]
        min_x = int(np.min(roi_coords[:, 0]))
        min_y = int(np.min(roi_coords[:, 1]))
        max_x = int(np.max(roi_coords[:, 0])) + self.patch_size
        max_y = int(np.max(roi_coords[:, 1])) + self.patch_size
        return {
            'min_x': min_x, 'min_y': min_y,
            'max_x': max_x, 'max_y': max_y,
            'width': max_x - min_x, 'height': max_y - min_y
        }
    
    def extract_roi_from_wsi(self, wsi_path, bbox, output_path):
        """
        Extract ROI, scale proportionally to fit in target_size, add padding.
        
        Steps:
        1. Read region from WSI
        2. Scale proportionally (same ratio for width and height) to fit target_size
        3. Add padding (black) to make exactly target_size x target_size
        """
        self.load_wsi(wsi_path)
        
        # Clamp coordinates
        x = max(0, bbox['min_x'])
        y = max(0, bbox['min_y'])
        wsi_w, wsi_h = self.slide.dimensions
        width = min(bbox['width'], wsi_w - x)
        height = min(bbox['height'], wsi_h - y)
        
        # Read region at level 0
        roi_image = self.slide.read_region(
            location=(x, y),
            level=0,
            size=(width, height)
        ).convert('RGB')
        
        original_size = roi_image.size  # (width, height)
        
        # Calculate scale factor to fit in target_size (proportional)
        scale = min(self.target_size / width, self.target_size / height)
        
        # New size after proportional scaling
        new_width = int(width * scale)
        new_height = int(height * scale)
        
        # Resize proportionally (LANCZOS for quality)
        roi_resized = roi_image.resize((new_width, new_height), Image.LANCZOS)
        
        # Create target canvas with padding (black background)
        final_image = Image.new('RGB', (self.target_size, self.target_size), (0, 0, 0))
        
        # Paste resized image at top-left (could center if preferred)
        final_image.paste(roi_resized, (0, 0))
        
        # Save
        final_image.save(output_path)
        
        return {
            'original_width': original_size[0],
            'original_height': original_size[1],
            'scale_factor': scale,
            'scaled_width': new_width,
            'scaled_height': new_height,
            'final_size': self.target_size
        }
    
    def process_all_rois(self, wsi_path, coords, rois, output_dir, slide_name):
        print(f"Extracting ROI images from WSI for {slide_name}...")
        roi_images_dir = os.path.join(output_dir, slide_name, 'roi_images')
        
        for roi in tqdm(rois):
            roi_id = roi['roi_id']
            patch_indices = roi['patch_indices']
            
            bbox = self.compute_bounding_box(coords, patch_indices)
            roi['bounding_box'] = bbox
            
            output_path = os.path.join(roi_images_dir, f'roi_{roi_id}.png')
            try:
                scale_info = self.extract_roi_from_wsi(wsi_path, bbox, output_path)
                roi['image_path'] = f'roi_images/roi_{roi_id}.png'
                roi['scale_info'] = scale_info
            except Exception as e:
                print(f"Error extracting ROI {roi_id}: {e}")
                roi['image_path'] = None
                roi['extraction_error'] = str(e)
        
        print(f"Extracted {len(rois)} ROI images ({self.target_size}x{self.target_size} with padding)")
        return rois
    
    def close(self):
        if self.slide is not None:
            self.slide.close()
            self.slide = None

## 7. Graph Builder

In [ ]:
class HESTGraphBuilder:
    def __init__(self, output_dir):
        self.output_dir = output_dir

    def generate_rois(self, coords, labels, slide_name, threshold, min_patches=2):
        print(f"Generating ROIs (min {min_patches} patches per ROI)...")
        rois = []
        roi_id = 0
        lonely_patches = 0
        for cid in np.unique(labels):
            idxs = np.where(labels == cid)[0]
            if len(idxs) < 1: continue
            G = nx.Graph()
            G.add_nodes_from(range(len(idxs)))
            G.add_edges_from(cKDTree(coords[idxs]).query_pairs(r=threshold))
            for comp in nx.connected_components(G):
                orig_idxs = idxs[list(comp)].tolist()
                if len(orig_idxs) >= min_patches:
                    rois.append({
                        'roi_id': roi_id,
                        'cluster_id': int(cid),
                        'centroid': np.mean(coords[orig_idxs], axis=0).tolist(),
                        'num_patches': len(orig_idxs),
                        'patch_indices': orig_idxs
                    })
                    roi_id += 1
                else:
                    lonely_patches += len(orig_idxs)
        print(f"Generated {len(rois)} ROIs, {lonely_patches} lonely patches excluded")
        return rois
    
    def save_rois(self, rois, slide_name):
        rois_path = os.path.join(self.output_dir, slide_name, 'graphs', 'rois.json')
        with open(rois_path, 'w') as f:
            json.dump(rois, f, indent=2)

    def create_knowledge_graph(self, coords, labels, rois, patch_to_spot, adata, markers, analyzer, slide_name):
        print(f"Building Hierarchical Knowledge Graph (Cluster -> ROI -> Patch)...")
        G = nx.Graph()
        for cid in np.unique(labels):
            G.add_node(f"Cluster_{cid}", type='Cluster', level=0)
        patch_to_roi = {}
        for roi in rois:
            rid = roi['roi_id']
            cid = roi['cluster_id']
            bbox = roi.get('bounding_box', {})
            scale_info = roi.get('scale_info', {})
            G.add_node(f"ROI_{rid}", type='ROI', cluster_id=cid, size=roi['num_patches'], level=1,
                       centroid=roi['centroid'], bounding_box=bbox, scale_info=scale_info,
                       image_path=roi.get('image_path', ''))
            G.add_edge(f"ROI_{rid}", f"Cluster_{cid}", relation='part_of_cluster')
            for idx in roi['patch_indices']:
                patch_to_roi[idx] = rid
        for i in range(len(coords)):
            cid = int(labels[i])
            rid = patch_to_roi.get(i, None)
            attrs = {'type': 'Patch', 'x': int(coords[i,0]), 'y': int(coords[i,1]),
                     'cluster_id': cid, 'roi_id': rid, 'level': 2}
            if i in patch_to_spot:
                attrs['markers'] = analyzer.get_marker_expression(adata, patch_to_spot[i], markers)
            G.add_node(f"Patch_{i}", **attrs)
            if rid is not None:
                G.add_edge(f"Patch_{i}", f"ROI_{rid}", relation='part_of_roi')
            G.add_edge(f"Patch_{i}", f"Cluster_{cid}", relation='belongs_to_cluster')
        print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        with open(os.path.join(self.output_dir, slide_name, 'graphs', 'knowledge_graph.json'), 'w') as f:
            json.dump(nx.node_link_data(G), f, indent=2)
        return G

## 8. Main Pipeline

In [ ]:
loader = HESTDataLoader(CONFIG['DATA_DIR'])
visualizer = HESTVisualizer()
extractor = HESTFeatureExtractor(CONFIG['DEVICE'])
clusterer = HESTClusterer(n_clusters=CONFIG['N_CLUSTERS'])
graph_builder = HESTGraphBuilder(CONFIG['OUTPUT_DIR'])
analyzer = HESTTranscriptomicsAnalyzer(CONFIG['BREAST_CANCER_MARKERS'])
roi_extractor = HESTROIExtractor(patch_size=CONFIG['PATCH_SIZE'], target_size=CONFIG['ROI_TARGET_SIZE'])

for slide in CONFIG['SLIDES']:
    print(f"\n{'='*20} {slide} {'='*20}")
    
    coords_h5, barcodes = loader.load_patches_metadata(slide)
    adata = loader.load_transcriptomics(slide)
    visualizer.plot_spatial_spots(adata, slide)
    
    markers = analyzer.find_available_markers(adata)
    patch_to_spot = analyzer.map_patches_to_spots(barcodes, adata)
    
    h5_path = loader.get_patches_path(slide)
    wsi_path = loader.get_wsi_path(slide)
    features, coords = extractor.extract(h5_path, CONFIG['OUTPUT_DIR'], slide)
    
    if features is not None:
        labels = clusterer.run(features)
        emb, emb_labels = clusterer.compute_umap(features, labels)
        visualizer.plot_umap(emb, emb_labels, slide)
        visualizer.plot_clusters_spatial(coords, labels, slide)
        
        rois = graph_builder.generate_rois(coords, labels, slide, CONFIG['ROI_DISTANCE_THRESHOLD'],
                                            min_patches=CONFIG['MIN_PATCHES_PER_ROI'])
        
        rois = roi_extractor.process_all_rois(wsi_path, coords, rois, CONFIG['OUTPUT_DIR'], slide)
        
        graph_builder.save_rois(rois, slide)
        visualizer.plot_rois_spatial(coords, rois, slide)
        
        G = graph_builder.create_knowledge_graph(coords, labels, rois, patch_to_spot, adata, markers, analyzer, slide)
        
        patches_in_rois = sum(r['num_patches'] for r in rois)
        lonely = len(coords) - patches_in_rois
        print(f"\nSummary: {len(coords)} patches, {len(rois)} ROIs, {patches_in_rois} in ROIs, {lonely} lonely")

roi_extractor.close()


==================== TENX96 ====================
[TENX96] Patches: 6177
[TENX96] AnnData: (7233, 541)


Found 7/11 markers
Mapped 6177/6177 patches to spots
[TENX96] Loading cached features
Running KMeans with k=8...
Computing UMAP...


/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Generating ROIs (min 8 patches per ROI)...
Generated 75 ROIs, 664 lonely patches excluded
Extracting ROI images from WSI for TENX96...


  0%|          | 0/75 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Generating ROIs (min 8 patches per ROI)...
Generated 262 ROIs, 3530 lonely patches excluded
Extracting ROI images from WSI for TENX99...


  0%|          | 0/262 [00:00<?, ?it/s]

Loading WSI with OpenSlide: ../hest_data/wsis/TENX99.tif
WSI dimensions: (51351, 107121)


  1%|          | 3/262 [00:05<06:28,  1.50s/it]